# 🧱 Semana 11-12: Frameworks Web, Bibliotecas Avanzadas y APIs Externas

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| 🏗️ Django vs Flask | Cuándo usar cada uno, estructura de proyecto Django |
| 🔧 Flask Avanzado | Blueprints, extensiones, middleware, configuración |
| 📦 Bibliotecas avanzadas | `collections`, `itertools`, `functools`, `dataclasses` |
| 🌐 APIs externas | `requests` avanzado, OAuth, rate limiting, reintentos |
| 🗄️ SQLAlchemy avanzado | Relaciones complejas, consultas avanzadas, eventos |

---

> 💡 **Prerequisito:** Semanas 1-10 completadas.  
> **Instalaciones necesarias:**
> ```bash
> pip install flask flask-sqlalchemy requests
> ```

---
## 🏗️ Bloque 1: Django vs Flask — Elegir el Framework Correcto

### 📘 Teoría

#### Filosofías opuestas

| | Flask | Django |
|---|---|---|
| **Filosofía** | Microframework — vos elegís cada componente | Batteries included — todo viene integrado |
| **Curva de aprendizaje** | Baja al principio, sube al escalar | Alta al principio, estable al escalar |
| **ORM** | Opcional (SQLAlchemy por separado) | Django ORM integrado |
| **Admin** | No incluido | Panel de administración automático |
| **Autenticación** | Manual o extensión | Sistema completo incluido |
| **Ideal para** | APIs, microservicios, prototipos | Aplicaciones web completas, CMS |

#### Estructura típica de proyecto

```
Flask (simple)                Django (completo)
──────────────                ────────────────────────────
mi_app/                       mi_proyecto/
├── app.py                    ├── manage.py
├── models.py                 ├── mi_proyecto/
├── routes/                   │   ├── settings.py
│   ├── auth.py               │   └── urls.py
│   └── productos.py          ├── app_productos/
└── templates/                │   ├── models.py
                              │   ├── views.py
                              │   ├── urls.py
                              │   └── admin.py
                              └── templates/
```

#### Cuándo elegir cada uno

```
¿Necesitás admin automático?     → Django
¿Es una API REST pura?           → Flask / FastAPI
¿Proyecto grande con muchos devs?→ Django (convenciones claras)
¿Microservicio o prototipo?      → Flask
¿Sistema de autenticación complejo? → Django
¿Máximo control y flexibilidad?  → Flask
```

### 💡 Ejemplo resuelto 1 — Flask con Blueprints (estructura escalable)

In [ ]:
from flask import Flask, Blueprint, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Boolean
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import json, hashlib

# ✅ Flask estructurado con Blueprints — simula un proyecto real

# ── Modelos ──
class Base(DeclarativeBase): pass

class Usuario(Base):
    __tablename__ = 'usuarios'
    id       = Column(Integer, primary_key=True, autoincrement=True)
    username = Column(String(50), unique=True, nullable=False)
    email    = Column(String(100), unique=True, nullable=False)
    password_hash = Column(String(64))
    es_admin = Column(Boolean, default=False)
    productos = relationship('Producto', back_populates='creador')

    @staticmethod
    def hashear(pwd): return hashlib.sha256(pwd.encode()).hexdigest()
    def verificar(self, pwd): return self.password_hash == self.hashear(pwd)

class Producto(Base):
    __tablename__ = 'productos'
    id         = Column(Integer, primary_key=True, autoincrement=True)
    nombre     = Column(String(100), nullable=False)
    precio     = Column(Float, nullable=False)
    stock      = Column(Integer, default=0)
    creador_id = Column(Integer, ForeignKey('usuarios.id'))
    creador    = relationship('Usuario', back_populates='productos')

    def to_dict(self):
        return {'id': self.id, 'nombre': self.nombre,
                'precio': self.precio, 'stock': self.stock,
                'creador_id': self.creador_id}

engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

# ── Helpers de autenticación ──
def autenticar(req):
    u, p = req.headers.get('X-User'), req.headers.get('X-Pass')
    if not u or not p: return None
    with Session(engine) as s:
        usuario = s.query(Usuario).filter_by(username=u).first()
        if usuario and usuario.verificar(p):
            s.expunge(usuario)
            return usuario
    return None

# ── Blueprint: autenticación ──
auth_bp = Blueprint('auth', __name__, url_prefix='/auth')

@auth_bp.route('/register', methods=['POST'])
def register():
    d = request.get_json() or {}
    campos = ['username', 'email', 'password']
    faltantes = [c for c in campos if not d.get(c)]
    if faltantes:
        return jsonify({'error': f'Faltan: {faltantes}'}), 400
    with Session(engine) as s:
        if s.query(Usuario).filter_by(username=d['username']).first():
            return jsonify({'error': 'Username ya existe'}), 409
        if s.query(Usuario).filter_by(email=d['email']).first():
            return jsonify({'error': 'Email ya registrado'}), 409
        nuevo = Usuario(username=d['username'], email=d['email'],
                        password_hash=Usuario.hashear(d['password']),
                        es_admin=d.get('es_admin', False))
        s.add(nuevo)
        s.commit()
        return jsonify({'id': nuevo.id, 'username': nuevo.username}), 201

@auth_bp.route('/me', methods=['GET'])
def me():
    usuario = autenticar(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    return jsonify({'username': usuario.username, 'email': usuario.email,
                    'es_admin': usuario.es_admin})

# ── Blueprint: productos ──
productos_bp = Blueprint('productos', __name__, url_prefix='/api/productos')

@productos_bp.route('', methods=['GET'])
def listar():
    with Session(engine) as s:
        prods = s.query(Producto).all()
        return jsonify([p.to_dict() for p in prods])

@productos_bp.route('', methods=['POST'])
def crear():
    usuario = autenticar(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    d = request.get_json() or {}
    if not d.get('nombre') or d.get('precio') is None:
        return jsonify({'error': 'nombre y precio son requeridos'}), 400
    if d['precio'] <= 0:
        return jsonify({'error': 'precio debe ser positivo'}), 400
    with Session(engine) as s:
        p = Producto(nombre=d['nombre'], precio=d['precio'],
                     stock=d.get('stock', 0), creador_id=usuario.id)
        s.add(p)
        s.commit()
        return jsonify(p.to_dict()), 201

@productos_bp.route('/<int:pid>', methods=['DELETE'])
def eliminar(pid):
    usuario = autenticar(request)
    if not usuario: return jsonify({'error': 'No autorizado'}), 401
    with Session(engine) as s:
        p = s.get(Producto, pid)
        if not p: return jsonify({'error': 'No encontrado'}), 404
        if p.creador_id != usuario.id and not usuario.es_admin:
            return jsonify({'error': 'Sin permiso para eliminar este producto'}), 403
        s.delete(p)
        s.commit()
        return '', 204

# ── Middleware — before/after request ──
def crear_app():
    app = Flask(__name__)
    app.register_blueprint(auth_bp)
    app.register_blueprint(productos_bp)

    @app.before_request
    def log_request():
        pass  # en producción: loguear cada request

    @app.after_request
    def agregar_headers(response):
        response.headers['X-API-Version'] = '1.0'
        response.headers['X-Powered-By']  = 'Python/Flask'
        return response

    @app.errorhandler(404)
    def not_found(e):
        return jsonify({'error': 'Endpoint no encontrado', 'codigo': 404}), 404

    @app.errorhandler(500)
    def server_error(e):
        return jsonify({'error': 'Error interno del servidor', 'codigo': 500}), 500

    return app

app = crear_app()

# ── Pruebas ──
print('=== Pruebas con Blueprints ===')
hj = {'Content-Type': 'application/json'}

with app.test_client() as c:
    # Registrar usuarios
    for user, email, pwd, admin in [
        ('ana',  'ana@mail.com',  'pass1', False),
        ('admin','admin@mail.com','admin1',True),
    ]:
        r = c.post('/auth/register',
                   data=json.dumps({'username':user,'email':email,'password':pwd,'es_admin':admin}),
                   headers=hj)
        print(f'  Registro {user}: {r.status_code}')

    # Consultar perfil
    r = c.get('/auth/me', headers={**hj,'X-User':'ana','X-Pass':'pass1'})
    print(f'  GET /auth/me: {r.get_json()}')

    # Crear productos
    auth_ana = {**hj,'X-User':'ana','X-Pass':'pass1'}
    auth_adm = {**hj,'X-User':'admin','X-Pass':'admin1'}
    for prod in [{'nombre':'Laptop','precio':1200,'stock':5},
                 {'nombre':'Mouse', 'precio':25,  'stock':20}]:
        r = c.post('/api/productos', data=json.dumps(prod), headers=auth_ana)
        print(f'  Crear {prod["nombre"]}: {r.status_code} → id={r.get_json()["id"]}')

    # Ana no puede borrar el producto de admin (inexistente aquí, verificamos 403)
    r = c.delete('/api/productos/2', headers=auth_adm)  # admin sí puede
    print(f'  Admin elimina producto #2: {r.status_code}')

    # Verificar headers de respuesta
    r = c.get('/api/productos')
    print(f'  X-API-Version: {r.headers.get("X-API-Version")}')
    print(f'  X-Powered-By:  {r.headers.get("X-Powered-By")}')

    # Error 404
    r = c.get('/ruta/inexistente')
    print(f'  Ruta inválida → {r.status_code}: {r.get_json()["error"]}')

### ✏️ Ejercicio guiado 1 — Agregar Blueprint de reportes

Agregá un tercer Blueprint `reportes_bp` a la app del ejemplo con los siguientes endpoints:

- `GET /api/reportes/resumen` → total de productos, stock total, precio promedio
- `GET /api/reportes/top?n=3` → los N productos más caros
- `GET /api/reportes/sin-stock` → productos con stock = 0

**Pistas:**
- Usá `url_prefix='/api/reportes'` en el Blueprint
- Para el top: `session.query(Producto).order_by(Producto.precio.desc()).limit(n).all()`
- Registrá el Blueprint con `app.register_blueprint(reportes_bp)`

In [ ]:
# ✏️ Definí y registrá el Blueprint de reportes:

reportes_bp = Blueprint('reportes', __name__, url_prefix='/api/reportes')

@reportes_bp.route('/resumen', methods=['GET'])
def resumen():
    pass  # Tu código aquí

@reportes_bp.route('/top', methods=['GET'])
def top_productos():
    n = int(request.args.get('n', 3))
    pass  # Tu código aquí

@reportes_bp.route('/sin-stock', methods=['GET'])
def sin_stock():
    pass  # Tu código aquí

# Registrá el blueprint y probá:
app.register_blueprint(reportes_bp)

with app.test_client() as c:
    r = c.get('/api/reportes/resumen')
    print(f'GET /api/reportes/resumen → {r.status_code}: {r.get_json()}')
    r = c.get('/api/reportes/top?n=2')
    print(f'GET /api/reportes/top?n=2 → {r.status_code}: {r.get_json()}')


<details>
<summary>👀 Ver solución</summary>

```python
@reportes_bp.route('/resumen', methods=['GET'])
def resumen():
    with Session(engine) as s:
        prods = s.query(Producto).all()
        total = len(prods)
        stock_total = sum(p.stock for p in prods)
        precio_prom = sum(p.precio for p in prods) / total if total else 0
        return jsonify({'total_productos': total,
                        'stock_total': stock_total,
                        'precio_promedio': round(precio_prom, 2)})

@reportes_bp.route('/top', methods=['GET'])
def top_productos():
    n = int(request.args.get('n', 3))
    with Session(engine) as s:
        prods = s.query(Producto).order_by(Producto.precio.desc()).limit(n).all()
        return jsonify([p.to_dict() for p in prods])

@reportes_bp.route('/sin-stock', methods=['GET'])
def sin_stock():
    with Session(engine) as s:
        prods = s.query(Producto).filter_by(stock=0).all()
        return jsonify([p.to_dict() for p in prods])
```
</details>

### 🚀 Ejercicio libre 1 — App completa con 4 Blueprints

Construí una app Flask de **gestión de una librería** con esta estructura:

- `auth_bp` → `/auth` — registro, login, perfil
- `libros_bp` → `/api/libros` — CRUD completo
- `autores_bp` → `/api/autores` — CRUD completo
- `reportes_bp` → `/api/reportes` — estadísticas

**Reglas de negocio:**
- Solo usuarios autenticados pueden crear/editar/borrar
- Solo el creador o un admin puede borrar un libro
- `GET /api/reportes/autores-con-mas-libros` → ranking de autores
- `GET /api/reportes/libros-por-genero` → conteo por género

In [ ]:
from flask import Flask, Blueprint, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Boolean
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import json, hashlib

# 🚀 Tu app de librería aquí:


---
## 📦 Bloque 2: Bibliotecas Estándar Avanzadas

### 📘 Teoría

Python viene con una biblioteca estándar muy rica. Estas son las más útiles para ciencia de datos y backend.

#### `collections` — Estructuras de datos especializadas
```python
from collections import Counter, defaultdict, OrderedDict, namedtuple, deque

Counter('banana')                    # {'a':3, 'n':2, 'b':1}
Counter([1,2,2,3,3,3]).most_common(2) # [(3,3),(2,2)]

dd = defaultdict(list)               # no lanza KeyError
dd['frutas'].append('manzana')

Punto = namedtuple('Punto', ['x','y'])  # tupla con nombres
p = Punto(3, 4)
p.x, p.y                             # 3, 4
```

#### `itertools` — Iteradores eficientes
```python
from itertools import chain, groupby, product, combinations, islice

list(chain([1,2], [3,4], [5]))        # [1,2,3,4,5]
list(combinations([1,2,3], 2))        # [(1,2),(1,3),(2,3)]
list(islice(range(1000), 5))          # [0,1,2,3,4] — eficiente
```

#### `functools` — Programación funcional
```python
from functools import reduce, lru_cache, partial

reduce(lambda a,b: a+b, [1,2,3,4])   # 10

@lru_cache(maxsize=128)               # memoización automática
def fibonacci(n):
    if n < 2: return n
    return fibonacci(n-1) + fibonacci(n-2)

doble = partial(lambda x,n: x*n, n=2)  # función con arg fijo
doble(5)  # 10
```

#### `dataclasses` — Clases de datos modernas
```python
from dataclasses import dataclass, field

@dataclass
class Producto:
    nombre: str
    precio: float
    stock: int = 0
    tags: list = field(default_factory=list)

    def con_descuento(self, pct):
        return self.precio * (1 - pct/100)

p = Producto('Laptop', 1200.0, 10)
print(p)  # Producto(nombre='Laptop', precio=1200.0, stock=10, tags=[])
```

### 💡 Ejemplo resuelto 2 — Usando bibliotecas estándar para análisis

In [ ]:
from collections import Counter, defaultdict, namedtuple
from itertools import groupby, chain, combinations
from functools import lru_cache, reduce
from dataclasses import dataclass, field
import time

# ✅ Casos de uso reales de cada biblioteca

# ─── collections.Counter ───
print('=== Counter: análisis de texto ===')
texto = 'el rapido zorro marron salta sobre el perro perezoso el zorro es rapido'
palabras = texto.split()
conteo = Counter(palabras)
print(f'  Palabras únicas: {len(conteo)}')
print(f'  Top 3 más frecuentes: {conteo.most_common(3)}')
print(f'  Veces que aparece "el": {conteo["el"]}')

# ─── collections.defaultdict ───
print('\n=== defaultdict: agrupar ventas por categoría ===')
ventas = [
    ('Electrónica', 1200), ('Ropa', 25), ('Electrónica', 89),
    ('Hogar', 350), ('Ropa', 60), ('Electrónica', 299),
]
por_categoria = defaultdict(list)
for cat, monto in ventas:
    por_categoria[cat].append(monto)
for cat, montos in sorted(por_categoria.items()):
    print(f'  {cat:15}: total=${sum(montos):,.0f}  items={len(montos)}')

# ─── namedtuple ───
print('\n=== namedtuple: datos estructurados ===')
Empleado = namedtuple('Empleado', ['nombre', 'depto', 'salario'])
equipo = [
    Empleado('Ana',    'IT',     85000),
    Empleado('Luis',   'Ventas', 62000),
    Empleado('Marta',  'IT',     91000),
    Empleado('Carlos', 'RRHH',   55000),
]
it_team = [e for e in equipo if e.depto == 'IT']
print(f'  Equipo IT: {[e.nombre for e in it_team]}')
print(f'  Salario promedio IT: ${sum(e.salario for e in it_team)/len(it_team):,.0f}')

# ─── lru_cache: memoización ───
print('\n=== lru_cache: fibonacci con y sin caché ===')

def fib_lento(n):
    if n < 2: return n
    return fib_lento(n-1) + fib_lento(n-2)

@lru_cache(maxsize=None)
def fib_rapido(n):
    if n < 2: return n
    return fib_rapido(n-1) + fib_rapido(n-2)

n = 30
t0 = time.perf_counter(); fib_lento(n);  t_lento  = time.perf_counter()-t0
t0 = time.perf_counter(); fib_rapido(n); t_rapido = time.perf_counter()-t0
print(f'  fib({n}) sin caché: {t_lento*1000:.1f} ms')
print(f'  fib({n}) con caché: {t_rapido*1000:.3f} ms')
print(f'  Mejora: {t_lento/t_rapido:.0f}x más rápido')

# ─── dataclass ───
print('\n=== dataclass: modelo de producto ===')

@dataclass(order=True)
class Producto:
    precio:   float
    nombre:   str  = field(compare=False)
    stock:    int  = field(default=0, compare=False)
    tags:     list = field(default_factory=list, compare=False)

    def con_descuento(self, pct: float) -> float:
        return round(self.precio * (1 - pct/100), 2)

    def esta_disponible(self) -> bool:
        return self.stock > 0

prods = [
    Producto(1200.0, 'Laptop',  10, ['tech', 'trabajo']),
    Producto(89.99,  'Auriculares', 50, ['tech', 'audio']),
    Producto(25.0,   'Cable USB', 0,  ['accesorios']),
]
print(f'  Productos: {[p.nombre for p in prods]}')
print(f'  Ordenados por precio: {[p.nombre for p in sorted(prods)]}')
print(f'  Disponibles: {[p.nombre for p in prods if p.esta_disponible()]}')
print(f'  Laptop con 20% descuento: ${prods[0].con_descuento(20)}')

# ─── itertools ───
print('\n=== itertools: combinaciones de productos ===')
nombres = ['Laptop', 'Mouse', 'Monitor']
combos = list(combinations(nombres, 2))
print(f'  Combos posibles de 2: {combos}')
total = reduce(lambda a, b: a + b, [p.precio for p in prods])
print(f'  Precio total (reduce): ${total:.2f}')

### ✏️ Ejercicio guiado 2 — Análisis con bibliotecas estándar

Usá `Counter`, `defaultdict` y `lru_cache` para analizar el siguiente dataset de logs.

**Pistas:**
- `Counter` es ideal para contar frecuencias
- `defaultdict(lambda: defaultdict(int))` crea dicts anidados automáticamente
- `lru_cache` es útil para funciones puras que se llaman repetidamente con los mismos args

In [ ]:
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import groupby

# Dataset: logs de acceso a una API
logs = [
    {'ip': '192.168.1.1', 'endpoint': '/productos',  'status': 200, 'ms': 45},
    {'ip': '10.0.0.2',    'endpoint': '/usuarios',   'status': 200, 'ms': 120},
    {'ip': '192.168.1.1', 'endpoint': '/productos',  'status': 200, 'ms': 38},
    {'ip': '172.16.0.5',  'endpoint': '/auth/login', 'status': 401, 'ms': 15},
    {'ip': '10.0.0.2',    'endpoint': '/productos',  'status': 404, 'ms': 8},
    {'ip': '192.168.1.1', 'endpoint': '/auth/login', 'status': 200, 'ms': 95},
    {'ip': '172.16.0.5',  'endpoint': '/auth/login', 'status': 401, 'ms': 12},
    {'ip': '10.0.0.2',    'endpoint': '/usuarios',   'status': 200, 'ms': 88},
    {'ip': '192.168.1.1', 'endpoint': '/productos',  'status': 200, 'ms': 52},
    {'ip': '172.16.0.5',  'endpoint': '/auth/login', 'status': 401, 'ms': 11},
]

# ✏️ Análisis 1: Top 3 endpoints más llamados con Counter
print('=== Top endpoints ===')
# contador_endpoints = Counter(log['endpoint'] for log in logs)


# ✏️ Análisis 2: Tasa de error (status != 200) por endpoint con defaultdict
print('\n=== Tasa de error por endpoint ===')
# errores = defaultdict(lambda: {'total': 0, 'errores': 0})


# ✏️ Análisis 3: IPs con más de 2 requests con status 401 (posibles ataques)
print('\n=== IPs sospechosas (> 2 intentos fallidos) ===')


# ✏️ Análisis 4: tiempo de respuesta promedio por endpoint
print('\n=== Tiempo promedio por endpoint ===')


<details>
<summary>👀 Ver solución</summary>

```python
# Análisis 1
contador = Counter(log['endpoint'] for log in logs)
for ep, count in contador.most_common(3):
    print(f'  {ep}: {count} llamadas')

# Análisis 2
errores = defaultdict(lambda: {'total': 0, 'errores': 0})
for log in logs:
    errores[log['endpoint']]['total'] += 1
    if log['status'] != 200:
        errores[log['endpoint']]['errores'] += 1
for ep, data in errores.items():
    tasa = data['errores'] / data['total'] * 100
    print(f'  {ep}: {tasa:.0f}% errores')

# Análisis 3
fallos_por_ip = Counter(log['ip'] for log in logs if log['status'] != 200)
sospechosas = {ip: c for ip, c in fallos_por_ip.items() if c > 2}
print(f'  {sospechosas}')

# Análisis 4
tiempos = defaultdict(list)
for log in logs:
    tiempos[log['endpoint']].append(log['ms'])
for ep, ms in tiempos.items():
    print(f'  {ep}: {sum(ms)/len(ms):.1f} ms promedio')
```
</details>

### 🚀 Ejercicio libre 2 — Sistema de caché con decoradores

Implementá un decorador `@cache_resultado(ttl_segundos=60)` que:

1. Guarde el resultado de una función en un diccionario con timestamp
2. Si se llama de nuevo con los mismos argumentos y el TTL no expiró, retorne el valor cacheado
3. Si expiró, recalcule y actualice el caché
4. Exponga métodos `.cache_info()` (cuántos hits/misses) y `.cache_clear()`

Usalo para cachear una función que simula una consulta lenta a la base de datos.

**Pistas:**
- Un decorador es una función que recibe una función y retorna una función
- Usá `time.time()` para timestamps
- Los argumentos como clave del caché: `str(args) + str(kwargs)`

In [ ]:
import time
from functools import wraps

# 🚀 Tu decorador de caché aquí:

def cache_resultado(ttl_segundos=60):
    def decorador(func):
        pass  # Tu implementación
    return decorador

# Test:
@cache_resultado(ttl_segundos=2)
def consulta_lenta(user_id):
    time.sleep(0.1)  # simula consulta a BD
    return {'user_id': user_id, 'nombre': f'Usuario_{user_id}'}

# Probá: primera llamada lenta, segunda rápida, después de 2s lenta de nuevo


---
## 🌐 Bloque 3: APIs Externas — Requests Avanzado

### 📘 Teoría

#### Sesiones y reintentos
```python
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Session reutiliza conexiones TCP — más eficiente
session = requests.Session()

# Configurar reintentos automáticos
retry = Retry(total=3, backoff_factor=1,
              status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry)
session.mount('https://', adapter)

# Timeout siempre (connect_timeout, read_timeout)
response = session.get(url, timeout=(5, 30))
```

#### Manejo robusto de errores
```python
try:
    r = session.get(url, timeout=10)
    r.raise_for_status()  # lanza HTTPError si status >= 400
    datos = r.json()
except requests.exceptions.ConnectionError:
    print('Sin conexión a internet')
except requests.exceptions.Timeout:
    print('La solicitud tardó demasiado')
except requests.exceptions.HTTPError as e:
    print(f'Error HTTP: {e.response.status_code}')
except requests.exceptions.JSONDecodeError:
    print('La respuesta no es JSON válido')
```

#### Rate Limiting
Muchas APIs limitan la cantidad de requests por minuto/hora.
```python
import time

def llamar_con_limite(urls, requests_por_segundo=2):
    intervalo = 1 / requests_por_segundo
    for url in urls:
        response = requests.get(url)
        yield response
        time.sleep(intervalo)  # respetar el límite
```

### 💡 Ejemplo resuelto 3 — Cliente robusto para API con reintentos

In [ ]:
import requests
import time
import json
from typing import Optional, Dict, Any

# ✅ Cliente HTTP robusto con reintentos, caché y manejo de errores

class ClienteAPI:
    """
    Cliente HTTP reutilizable con:
    - Sesión persistente (reutiliza conexiones)
    - Timeout configurable
    - Reintentos con backoff exponencial
    - Caché en memoria
    - Logging de requests
    """

    def __init__(self, base_url: str, timeout: int = 10,
                 max_reintentos: int = 3, cache_ttl: int = 300):
        self.base_url      = base_url.rstrip('/')
        self.timeout       = timeout
        self.max_reintentos= max_reintentos
        self.cache_ttl     = cache_ttl
        self._cache        = {}
        self._stats        = {'requests': 0, 'cache_hits': 0, 'errores': 0}
        self.session       = requests.Session()
        self.session.headers.update({'User-Agent': 'MiApp/1.0 Python'})

    def _cache_get(self, key: str) -> Optional[Any]:
        if key in self._cache:
            valor, timestamp = self._cache[key]
            if time.time() - timestamp < self.cache_ttl:
                return valor
            del self._cache[key]
        return None

    def _cache_set(self, key: str, valor: Any):
        self._cache[key] = (valor, time.time())

    def get(self, endpoint: str, params: dict = None,
            usar_cache: bool = True) -> Optional[Any]:
        url       = f'{self.base_url}/{endpoint.lstrip("/")}'
        cache_key = f'{url}?{params}'

        if usar_cache:
            cached = self._cache_get(cache_key)
            if cached is not None:
                self._stats['cache_hits'] += 1
                return cached

        for intento in range(self.max_reintentos):
            try:
                self._stats['requests'] += 1
                r = self.session.get(url, params=params, timeout=self.timeout)
                r.raise_for_status()
                datos = r.json()
                if usar_cache:
                    self._cache_set(cache_key, datos)
                return datos

            except requests.exceptions.HTTPError as e:
                codigo = e.response.status_code
                if codigo == 404:
                    return None  # no reintentar si no existe
                if codigo == 429:  # rate limit
                    espera = 2 ** intento
                    print(f'  Rate limit, esperando {espera}s...')
                    time.sleep(espera)
                self._stats['errores'] += 1

            except requests.exceptions.ConnectionError:
                self._stats['errores'] += 1
                if intento < self.max_reintentos - 1:
                    time.sleep(2 ** intento)

            except requests.exceptions.Timeout:
                self._stats['errores'] += 1
                print(f'  Timeout en intento {intento+1}')

        return None

    def post(self, endpoint: str, datos: dict) -> Optional[Any]:
        url = f'{self.base_url}/{endpoint.lstrip("/")}'
        try:
            self._stats['requests'] += 1
            r = self.session.post(url, json=datos, timeout=self.timeout)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.RequestException as e:
            self._stats['errores'] += 1
            print(f'  Error POST: {e}')
            return None

    def stats(self) -> dict:
        total   = self._stats['requests']
        hits    = self._stats['cache_hits']
        errores = self._stats['errores']
        hit_rate = hits / (total + hits) * 100 if (total + hits) > 0 else 0
        return {**self._stats, 'cache_hit_rate': f'{hit_rate:.1f}%'}


# ── Usar el cliente con JSONPlaceholder ──
cliente = ClienteAPI('https://jsonplaceholder.typicode.com', cache_ttl=60)

print('=== Requests con caché ===')

# Primera vez: va a la red
t0 = time.perf_counter()
usuarios = cliente.get('/users')
t1 = time.perf_counter() - t0
print(f'  Primera llamada /users: {t1*1000:.0f} ms (red)')
if usuarios:
    print(f'  Usuarios obtenidos: {len(usuarios)}')

# Segunda vez: desde caché
t0 = time.perf_counter()
usuarios2 = cliente.get('/users')
t2 = time.perf_counter() - t0
print(f'  Segunda llamada /users: {t2*1000:.2f} ms (caché)')
print(f'  Speedup: {t1/t2:.0f}x más rápido')

# Varios endpoints
print('\n=== Obtener posts de varios usuarios ===')
for user_id in [1, 2, 3]:
    posts = cliente.get('/posts', params={'userId': user_id})
    if posts:
        print(f'  Usuario {user_id}: {len(posts)} posts')

# POST
print('\n=== Crear un nuevo post ===')
nuevo = cliente.post('/posts', {'title': 'Mi post', 'body': 'Contenido', 'userId': 1})
if nuevo:
    print(f'  Post creado con ID: {nuevo["id"]}')

# Recurso no existente
print('\n=== Recurso inexistente ===')
resultado = cliente.get('/users/9999')
print(f'  /users/9999 → {resultado} (esperado None)')

# Stats
print(f'\n=== Estadísticas del cliente ===')
for k, v in cliente.stats().items():
    print(f'  {k}: {v}')

### ✏️ Ejercicio guiado 3 — Procesar múltiples endpoints en paralelo

Usando el `ClienteAPI` del ejemplo, obtené datos de múltiples usuarios **en secuencia** y analizalos.

**Pistas:**
- Para obtener todos los posts y todos los usuarios: dos llamadas separadas
- Cruzá los datos usando diccionarios para eficiencia: `{user['id']: user for user in usuarios}`
- `Counter` es útil para contar posts por usuario

In [ ]:
from collections import Counter

# (Requiere haber ejecutado la celda anterior — cliente disponible)

# ✏️ Análisis 1: usuario con más posts
print('=== Usuario con más posts ===')
# posts = cliente.get('/posts')
# usuarios = cliente.get('/users')
# Tu código aquí:


# ✏️ Análisis 2: título de post más largo por usuario
print('\n=== Post más largo por usuario (primeros 3) ===')
# Tu código aquí:


# ✏️ Análisis 3: usuarios ordenados por cantidad de posts
print('\n=== Ranking de usuarios por cantidad de posts ===')
# Tu código aquí:


<details>
<summary>👀 Ver solución</summary>

```python
posts    = cliente.get('/posts')
usuarios = cliente.get('/users')
user_map = {u['id']: u['name'] for u in usuarios}

# Análisis 1
conteo = Counter(p['userId'] for p in posts)
user_id_max, cantidad = conteo.most_common(1)[0]
print(f'  {user_map[user_id_max]}: {cantidad} posts')

# Análisis 2
from itertools import groupby
posts_por_user = {}
for p in posts:
    posts_por_user.setdefault(p['userId'], []).append(p)
for uid, ps in list(posts_por_user.items())[:3]:
    mas_largo = max(ps, key=lambda p: len(p['title']))
    print(f'  {user_map[uid]}: "{mas_largo["title"][:40]}..."')

# Análisis 3
for uid, count in conteo.most_common():
    print(f'  {user_map[uid]:25} {count} posts')
```
</details>

### 🚀 Ejercicio libre 3 — Cliente para API climática

Usá la API gratuita de Open Meteo (sin clave) para analizar el clima:

**URL base:** `https://api.open-meteo.com/v1/forecast`

**Parámetros básicos:**
```
latitude=-34.61&longitude=-58.38  # Buenos Aires
&daily=temperature_2m_max,temperature_2m_min,precipitation_sum
&timezone=America/Argentina/Buenos_Aires
&forecast_days=7
```

**Tu análisis debe incluir:**
1. Temperatura máxima y mínima de los próximos 7 días
2. Día más caluroso y más frío de la semana
3. Total de precipitaciones esperadas
4. Comparación entre dos ciudades (ej: Buenos Aires vs Mendoza)
5. Manejo robusto de errores usando tu `ClienteAPI`

In [ ]:
import requests
import time

# 🚀 Tu análisis climático aquí:


---
## 🗄️ Bloque 4: SQLAlchemy Avanzado

### 📘 Teoría

#### Relaciones avanzadas
```python
# Many-to-Many con tabla de asociación que tiene atributos extra
class Inscripcion(Base):
    __tablename__ = 'inscripciones'
    alumno_id  = Column(Integer, ForeignKey('alumnos.id'), primary_key=True)
    curso_id   = Column(Integer, ForeignKey('cursos.id'),  primary_key=True)
    fecha      = Column(String)
    nota_final = Column(Float, nullable=True)
    alumno     = relationship('Alumno', back_populates='inscripciones')
    curso      = relationship('Curso',  back_populates='inscripciones')
```

#### Consultas avanzadas con ORM
```python
from sqlalchemy import func, and_, or_, desc

# Funciones de agregación
session.query(func.count(Producto.id)).scalar()
session.query(func.avg(Producto.precio)).scalar()

# Filtros compuestos
session.query(Producto).filter(
    and_(Producto.precio > 100, Producto.stock > 0)
).all()

# JOIN explícito
session.query(Producto, Categoria).join(
    Categoria, Producto.categoria_id == Categoria.id
).filter(Categoria.nombre == 'Electrónica').all()

# Subquery
precio_promedio = session.query(func.avg(Producto.precio)).scalar_subquery()
caros = session.query(Producto).filter(Producto.precio > precio_promedio).all()
```

#### Eventos
```python
from sqlalchemy import event

@event.listens_for(Producto, 'before_insert')
def antes_de_insertar(mapper, connection, target):
    target.nombre = target.nombre.strip().title()  # normalizar
```

### 💡 Ejemplo resuelto 4 — Consultas avanzadas y eventos

In [ ]:
from sqlalchemy import (
    create_engine, Column, Integer, String, Float, ForeignKey,
    func, and_, or_, desc, event
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session

# ✅ SQLAlchemy avanzado: consultas complejas y eventos

class Base(DeclarativeBase): pass

class Categoria(Base):
    __tablename__ = 'categorias'
    id       = Column(Integer, primary_key=True, autoincrement=True)
    nombre   = Column(String(50), nullable=False, unique=True)
    productos = relationship('Producto', back_populates='categoria')

class Producto(Base):
    __tablename__ = 'productos'
    id           = Column(Integer, primary_key=True, autoincrement=True)
    nombre       = Column(String(100), nullable=False)
    precio       = Column(Float, nullable=False)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey('categorias.id'))
    categoria    = relationship('Categoria', back_populates='productos')

class Alumno(Base):
    __tablename__ = 'alumnos'
    id     = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String(100), nullable=False)
    inscripciones = relationship('Inscripcion', back_populates='alumno')

class Curso(Base):
    __tablename__ = 'cursos'
    id     = Column(Integer, primary_key=True, autoincrement=True)
    nombre = Column(String(100), nullable=False)
    precio = Column(Float)
    inscripciones = relationship('Inscripcion', back_populates='curso')

class Inscripcion(Base):
    __tablename__ = 'inscripciones'
    alumno_id  = Column(Integer, ForeignKey('alumnos.id'), primary_key=True)
    curso_id   = Column(Integer, ForeignKey('cursos.id'),  primary_key=True)
    nota_final = Column(Float, nullable=True)
    alumno     = relationship('Alumno', back_populates='inscripciones')
    curso      = relationship('Curso',  back_populates='inscripciones')

# ── Evento: normalizar nombre antes de insertar ──
@event.listens_for(Producto, 'before_insert')
def normalizar_nombre(mapper, connection, target):
    target.nombre = target.nombre.strip().title()

engine = create_engine('sqlite:///:memory:', echo=False)
Base.metadata.create_all(engine)

with Session(engine) as s:
    # Datos
    cats = {n: Categoria(nombre=n) for n in ['Electrónica','Ropa','Hogar']}
    s.add_all(cats.values()); s.flush()

    prods = [
        Producto(nombre='  laptop pro  ', precio=1200, stock=10, categoria_id=cats['Electrónica'].id),
        Producto(nombre='mouse usb',      precio=25,   stock=50, categoria_id=cats['Electrónica'].id),
        Producto(nombre='AURICULARES',    precio=89,   stock=30, categoria_id=cats['Electrónica'].id),
        Producto(nombre='camiseta',       precio=25,   stock=0,  categoria_id=cats['Ropa'].id),
        Producto(nombre='sillon',         precio=350,  stock=5,  categoria_id=cats['Hogar'].id),
    ]
    s.add_all(prods); s.flush()

    alumnos = [Alumno(nombre=n) for n in ['Ana','Luis','Marta','Carlos']]
    cursos  = [Curso(nombre=n, precio=p) for n,p in
               [('Python Básico',500),('SQL Avanzado',800),('Data Science',1200)]]
    s.add_all(alumnos + cursos); s.flush()

    inscripciones = [
        Inscripcion(alumno_id=alumnos[0].id, curso_id=cursos[0].id, nota_final=9.5),
        Inscripcion(alumno_id=alumnos[0].id, curso_id=cursos[1].id, nota_final=8.0),
        Inscripcion(alumno_id=alumnos[1].id, curso_id=cursos[0].id, nota_final=7.0),
        Inscripcion(alumno_id=alumnos[2].id, curso_id=cursos[2].id, nota_final=None),
        Inscripcion(alumno_id=alumnos[3].id, curso_id=cursos[0].id, nota_final=6.5),
        Inscripcion(alumno_id=alumnos[3].id, curso_id=cursos[2].id, nota_final=9.0),
    ]
    s.add_all(inscripciones)
    s.commit()

    # ── Verificar evento de normalización ──
    print('=== Nombres normalizados (evento before_insert) ===')
    for p in s.query(Producto).all():
        print(f'  {p.nombre}')

    # ── Consultas avanzadas ──
    print('\n=== Funciones de agregación ===')
    total    = s.query(func.count(Producto.id)).scalar()
    prom     = s.query(func.avg(Producto.precio)).scalar()
    max_prec = s.query(func.max(Producto.precio)).scalar()
    print(f'  Total productos: {total}')
    print(f'  Precio promedio: ${prom:.2f}')
    print(f'  Precio máximo:   ${max_prec:.2f}')

    print('\n=== Productos sobre el precio promedio ===')
    precio_prom = s.query(func.avg(Producto.precio)).scalar_subquery()
    caros = s.query(Producto).filter(Producto.precio > precio_prom).all()
    for p in caros:
        print(f'  {p.nombre:20} ${p.precio:.2f}')

    print('\n=== Productos en stock por categoría ===')
    resultado = (
        s.query(Categoria.nombre, func.count(Producto.id), func.sum(Producto.stock))
        .join(Producto, Categoria.id == Producto.categoria_id)
        .filter(Producto.stock > 0)
        .group_by(Categoria.nombre)
        .all()
    )
    for cat, cant, stock in resultado:
        print(f'  {cat:15}: {cant} productos, {stock} unidades en stock')

    print('\n=== Alumnos con nota promedio > 7 ===')
    buenos = (
        s.query(Alumno.nombre, func.avg(Inscripcion.nota_final))
        .join(Inscripcion, Alumno.id == Inscripcion.alumno_id)
        .filter(Inscripcion.nota_final.isnot(None))
        .group_by(Alumno.nombre)
        .having(func.avg(Inscripcion.nota_final) > 7)
        .order_by(desc(func.avg(Inscripcion.nota_final)))
        .all()
    )
    for nombre, prom in buenos:
        print(f'  {nombre:10}: promedio {prom:.2f}')

### ✏️ Ejercicio guiado 4 — Consultas con OR, AND y subqueries

Usando la BD del ejemplo anterior, completá las consultas.

**Pistas:**
- `or_(condicion1, condicion2)` para condiciones alternativas
- `and_(condicion1, condicion2)` para condiciones simultáneas
- `.scalar_subquery()` convierte una consulta en subquery

In [ ]:
# (Requiere la BD del ejemplo anterior)

with Session(engine) as s:

    # ✏️ Consulta 1: productos baratos (< $50) O sin stock
    print('=== Productos baratos O sin stock ===')
    # resultado = s.query(Producto).filter(or_(...)).all()


    # ✏️ Consulta 2: alumnos inscriptos en MÁS DE 1 curso
    print('\n=== Alumnos con más de 1 inscripción ===')
    # Pista: GROUP BY + HAVING COUNT > 1


    # ✏️ Consulta 3: cursos sin ningún alumno inscripto
    print('\n=== Cursos sin alumnos ===')
    # Pista: LEFT JOIN + filter(Inscripcion.alumno_id == None)


    # ✏️ Consulta 4: el alumno con la nota más alta en cada curso
    print('\n=== Mejor nota por curso ===')
    # Pista: GROUP BY curso_id + func.max(nota_final)


<details>
<summary>👀 Ver solución</summary>

```python
# Consulta 1
resultado = s.query(Producto).filter(
    or_(Producto.precio < 50, Producto.stock == 0)
).all()
for p in resultado: print(f'  {p.nombre} ${p.precio} stock={p.stock}')

# Consulta 2
resultado = (
    s.query(Alumno.nombre, func.count(Inscripcion.curso_id).label('cursos'))
    .join(Inscripcion, Alumno.id == Inscripcion.alumno_id)
    .group_by(Alumno.nombre)
    .having(func.count(Inscripcion.curso_id) > 1)
    .all()
)

# Consulta 3
resultado = (
    s.query(Curso)
    .outerjoin(Inscripcion, Curso.id == Inscripcion.curso_id)
    .filter(Inscripcion.alumno_id == None)
    .all()
)

# Consulta 4
resultado = (
    s.query(Curso.nombre, func.max(Inscripcion.nota_final))
    .join(Inscripcion, Curso.id == Inscripcion.curso_id)
    .group_by(Curso.nombre)
    .all()
)
```
</details>

### 🚀 Ejercicio libre 4 — Sistema de e-learning completo

Construí un sistema de e-learning con SQLAlchemy ORM que incluya:

**Modelos:**
- `Instructor`: id, nombre, especialidad
- `Curso`: id, nombre, descripcion, precio, instructor_id, nivel ('básico'/'intermedio'/'avanzado')
- `Modulo`: id, curso_id, titulo, orden, duracion_min
- `Alumno`: id, nombre, email
- `Inscripcion`: alumno_id, curso_id, fecha, progreso (0-100), completado

**Eventos SQLAlchemy:**
- `before_insert` en Alumno: normalizar email a minúsculas
- `after_update` en Inscripcion: si progreso = 100, marcar completado = True automáticamente

**Consultas requeridas:**
1. Instructor con más alumnos en total
2. Alumnos con todos sus cursos completados
3. Curso con mayor ingreso (suma de precios de inscripciones)
4. Promedio de progreso por nivel de curso

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Boolean, event
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from sqlalchemy import func

# 🚀 Tu sistema de e-learning aquí:


---
## ✅ Resumen de la Semana 11-12

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| Flask Blueprints | Modularización, middleware, error handlers, headers |
| Bibliotecas estándar | `Counter`, `defaultdict`, `namedtuple`, `lru_cache`, `dataclass` |
| Requests avanzado | Sesiones, reintentos, caché, manejo robusto de errores |
| SQLAlchemy avanzado | `func`, `and_/or_`, subqueries, `having`, eventos |

### ➡️ Próximos pasos — Semana 13-14
- SQLAlchemy avanzado: interacción directa con BD
- Pruebas unitarias con `unittest` y `pytest`
- Cobertura de código con `coverage.py`

---

### 📚 Recursos recomendados
- [Flask Blueprints — Docs](https://flask.palletsprojects.com/en/latest/blueprints/)
- [Python collections — Docs](https://docs.python.org/3/library/collections.html)
- [Requests avanzado](https://requests.readthedocs.io/en/latest/user/advanced/)
- [SQLAlchemy ORM — Querying](https://docs.sqlalchemy.org/en/20/orm/queryguide/)